In [0]:
print(df.columns)

In [0]:
from pyspark.sql.functions import col, sum, avg, count, hour, to_date

df = spark.read.table("dev.silver.yellow_taxi_clean")

# 1. Total Trips and Revenue per Day
# Extracting the date and calculating count/sum of amount

daily_metrics = df.withColumn("date", to_date(col("pickup_datetime"))) \
    .groupBy("date") \
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue")
    ) \
    .orderBy("date")
daily_metrics.show()
daily_metrics.write.format("delta").mode("overwrite").saveAsTable("dev.gold.daily_metrics")

# 2. Average Trip Distance and Fare per Pickup Location
# Groups by pickup location ID and calculates averages
location_stats = df.groupBy("PU_Location_ID") \
    .agg(
        avg("trip_distance").alias("avg_distance"),
        avg("total_amount").alias("avg_fare")
    ) \
    .orderBy(col("avg_fare").desc())
location_stats.show()
location_stats.write.format("delta").mode("overwrite").saveAsTable("dev.gold.location_stats")

# 3. Peak Hours by Trip Volume
# Extracts the hour and counts trips
hourly_volume = df.withColumn("pickup_hour", hour(col("pickup_datetime"))) \
    .groupBy("pickup_hour") \
    .agg(count("*").alias("trip_count")) \
    .orderBy(col("trip_count").desc())
hourly_volume.show()
hourly_volume.write.format("delta").mode("overwrite").saveAsTable("dev.gold.hourly_volume")

# 4. Payment Type Breakdown
# Counts trips for each payment type
payment_breakdown = df.groupBy("payment_type") \
    .agg(count("*").alias("payment_type_count")) \
    .orderBy(col("payment_type_count").desc())
payment_breakdown.show()
payment_breakdown.write.format("delta").mode("overwrite").saveAsTable("dev.gold.payment_breakdown")
